# Unified Inference — GutBrainIE 2026 NER

Runs the **same** inference pipeline on every trained model checkpoint and saves one prediction file per model.


## 0. Imports & reproducibility

In [15]:
import json
import re
import copy
import numpy as np
import torch
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForTokenClassification

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

PyTorch: 2.12.1+cpu
CUDA available: False
Device: cpu


## 1. Label space (must match training)

In [16]:
ENTITY_LABELS = [
    "anatomical location",
    "animal",
    "bacteria",
    "biomedical technique",
    "chemical",
    "DDF",
    "dietary supplement",
    "drug",
    "food",
    "gene",
    "human",
    "microbiome",
    "statistical technique",
]

label_list = ["O"]
for lab in ENTITY_LABELS:
    label_list.append(f"B-{lab}")
    label_list.append(f"I-{lab}")

label2id = {k: v for v, k in enumerate(label_list)}
id2label  = {v: k for k, v in label2id.items()}

print(f"Total labels: {len(label_list)}")

Total labels: 27


## 2. Paths — edit here if your folder layout differs

In [17]:
# ── Repository root: the folder that contains both 'data' and 'src' ──────────
# The notebook lives at:  <PROJECT_ROOT>/src/ner/bert_NER_inference_all_models.ipynb
# so Path.cwd() is       <PROJECT_ROOT>/src/ner/  →  go up 2 levels
def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "data").exists() and (p / "src").exists():
            return p
    raise FileNotFoundError(
        f"Cannot find repo root (needs 'data' and 'src' subfolders). "
        f"Started from: {start}"
    )

PROJECT_ROOT = find_repo_root(Path.cwd())
MODELS_DIR   = PROJECT_ROOT / "src" / "ner" / "models"
PRED_DIR     = PROJECT_ROOT / "src" / "ner" / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

DEV_PATH = (
    PROJECT_ROOT
    / "data" / "GutBrainIE_Full_Collection_2026"
    / "Annotations" / "Dev" / "json_format" / "dev.json"
)

# ── Model checkpoints ─────────────────────────────────────────────────────────
MODELS = {
    #"NB1":  MODELS_DIR / "bert_biomedbert_ner_twopass_gold_silver",
    "NB1b": MODELS_DIR / "bert_biomedbert_ner_twopass_gold_silver_bronze",
    "NB1b_pubmed": MODELS_DIR / "pubmedbert_ner_twopass_gold_silver_bronze",
    # "NB2a": MODELS_DIR / "bert_ner_2026_curriculum_B1_gold",
    # "NB2b": MODELS_DIR / "bert_ner_2026_curriculum_B2_gold_plus_silver",
    # "NB3":  MODELS_DIR / "bert_biomedbert_ner_twopass_2025to2026_stageC_mix",
    # "NB4":  MODELS_DIR / "bert_biomedbert_ner_twopass_annotator_weights",
    # "NB4b":  MODELS_DIR / "bert_biomedbert_ner_twopass_annotator_weights_bronze",
}

PRED_FILES = {
    #"NB1":  PRED_DIR / "pred_NB1_gold_silver.json",
    "NB1b": PRED_DIR / "pred_NB1b_gold_silver_bronze.json",
    "NB1b_pubmed": PRED_DIR / "pred_NB1b_pubmed_gold_silver_bronze.json",
    # "NB2a": PRED_DIR / "pred_NB2a_curriculum_B1.json",
    # "NB2b": PRED_DIR / "pred_NB2b_curriculum_B2.json",
    # "NB3":  PRED_DIR / "pred_NB3_2526_stageC.json",
    # "NB4":  PRED_DIR / "pred_NB4_annotator_weights.json",
    # "NB4b":  PRED_DIR / "pred_NB4b_annotator_weights_bronze.json",
}

print("Project root :", PROJECT_ROOT)
print("Models dir   :", MODELS_DIR)
print("Predictions  :", PRED_DIR)
print("Dev file     :", DEV_PATH, "| exists:", DEV_PATH.exists())
print()
print("Model checkpoints:")
for tag, p in MODELS.items():
    status = "✓" if p.exists() else "✗ MISSING"
    print(f"  {status}  {tag:6s}  {p.relative_to(PROJECT_ROOT)}")

Project root : C:\Users\super\Documents\UniPd\ATA\SMTE-GutBrainIE
Models dir   : C:\Users\super\Documents\UniPd\ATA\SMTE-GutBrainIE\src\ner\models
Predictions  : C:\Users\super\Documents\UniPd\ATA\SMTE-GutBrainIE\src\ner\predictions
Dev file     : C:\Users\super\Documents\UniPd\ATA\SMTE-GutBrainIE\data\GutBrainIE_Full_Collection_2026\Annotations\Dev\json_format\dev.json | exists: True

Model checkpoints:
  ✓  NB1b    src\ner\models\bert_biomedbert_ner_twopass_gold_silver_bronze
  ✓  NB1b_pubmed  src\ner\models\pubmedbert_ner_twopass_gold_silver_bronze


## 3. Load dev data

In [18]:
with DEV_PATH.open(encoding="utf-8") as f:
    dev_data = json.load(f)

def prepare_documents(data: dict) -> list:
    """Split each article into title + abstract segments."""
    docs = []
    for pmid, article in data.items():
        meta = article.get("metadata", {})
        entities = article.get("entities", [])
        for loc in ("title", "abstract"):
            text = (meta.get(loc) or "").strip()
            if not text:
                continue
            docs.append({
                "pmid": str(pmid),
                "location": loc,
                "text": text,
                "entities": [e for e in entities if e.get("location") == loc],
            })
    return docs

dev_documents = prepare_documents(dev_data)
print(f"Dev articles: {len(dev_data)}")
print(f"Dev segments (title+abstract): {len(dev_documents)}")

Dev articles: 80
Dev segments (title+abstract): 160


## 4. Inference pipeline (shared by all models)

This is the **NB3 version** — the most complete:
- postprocess *before* threshold (fixes label remapping before filtering)
- `dedup_segment` + `soft_overlap_prune` after merge
- score = median token probability (more recall-friendly than mean)

In [19]:
# ── Thresholds (tuned on dev, from NB3 coordinate-ascent) ────────────────────
LABEL_THRESH_HIGH = {
    "DDF":                    0.88,
    "bacteria":               0.84,
    "statistical technique":  0.91,
    "biomedical technique":   0.78,
    "gene":                   0.68,
    "food":                   0.60,
    "chemical":               0.72,
    "dietary supplement":     0.78,
    "drug":                   0.80,
    "microbiome":             0.78,
    "anatomical location":    0.78,
    "human":                  0.70,
    "animal":                 0.70,
}

LABEL_THRESH_RECALL = {
    "food":                   0.43,
    "chemical":               0.65,
    "bacteria":               0.80,
    "dietary supplement":     0.72,
}
RECALL_LABELS  = {"chemical", "food", "bacteria", "dietary supplement"}
DEFAULT_THRESH = 0.80

print("Thresholds loaded.")

Thresholds loaded.


In [20]:
# ── FP filter lists ───────────────────────────────────────────────────────────
BAD_BACTERIA  = {"bacteria", "micro", "microbes", "microorganisms", "genera", "taxa"}
BAD_CHEMICAL  = {"metabolites", "neurotransmitters"}
BAD_DIETSUPP  = {"nnss", "sp", "fep", "ns9", "pro"}
BAD_MICROBIOME= {"micro", "microbiota", "gut"}
DIET_CONCEPT  = {
    "diet", "ketogenic diet", "high-fat diet", "high fat diet",
    "high glycemic diet", "vegetarian diet", "balanced diet",
    "western diet", "mediterranean diet",
}
BAD_FOOD_EXACT = {
    "control", "ketogenic", "high-fat", "high fat", "high",
    "glycemic index", "lycemic index", "food", "ingested food",
}

GENE_LIKE = re.compile(
    r"^(il-\d+|tnf(-?α)?|ifn(-?γ)?|tgf(-?β\d*)?|snca|park7|dj-1|mapt|apoe\d*|hla-[a-z0-9\*\:]+)$",
    re.IGNORECASE,
)
CHEM_LIKE = re.compile(
    r"(aβ|amyloid|scfa|gaba|succinate|butyrate|propionate|acetate|\b[a-z]+ate\b|\b[a-z]+acid\b|\(\d+\-\d+\))",
    re.IGNORECASE,
)
FOOD_ANCHORS = {
    "kefir", "yogurt", "milk", "cheese", "cookie",
    "lentil", "lentils", "buckwheat", "wheat", "rice", "tea", "coffee",
}
SUPP_HARD = {"capsule", "tablet", "extract", "powder"}
SUPP_SOFT = {"probiotic", "probiotics", "prebiotic", "prebiotics",
             "synbiotic", "synbiotics", "supplement"}
TRIM_CHARS = " \t\n\r.,;:()[]{}<>\"'"

print("Filter lists loaded.")

Filter lists loaded.


In [21]:
# ── Utility functions ─────────────────────────────────────────────────────────

def normalize_span(s: str) -> str:
    s = (s or "").strip().lower()
    return re.sub(r"\s+", " ", s)


def trim_entity_span(e: dict, text: str) -> dict:
    """Remove leading/trailing punctuation from a predicted span."""
    s   = max(0, min(int(e["start_idx"]), len(text)))
    end = max(0, min(int(e["end_idx"]),   len(text) - 1))
    while s <= end and text[s]   in TRIM_CHARS: s   += 1
    while end >= s and text[end] in TRIM_CHARS: end -= 1
    if s <= end:
        e["start_idx"] = s
        e["end_idx"]   = end
        e["text_span"] = text[s : end + 1]
    return e


def apply_simple_filters(entities: list) -> list:
    """Drop obvious false positives based on span text."""
    out = []
    for e in entities:
        s   = normalize_span(e.get("text_span", ""))
        lab = e["label"]
        if not s or len(s) <= 1:                                     continue
        if "<" in s or ">" in s:                                     continue
        if lab == "bacteria"          and s in BAD_BACTERIA:         continue
        if lab == "dietary supplement" and s in BAD_DIETSUPP:        continue
        if lab == "chemical"          and s in BAD_CHEMICAL:         continue
        if lab == "microbiome"        and s in BAD_MICROBIOME:       continue
        if lab == "food":
            if s in DIET_CONCEPT or s.endswith(" diet"): continue
            if s in BAD_FOOD_EXACT:                       continue
        out.append(e)
    return out


def postprocess_gene_vs_chemical(entities: list) -> list:
    """Remap chemical→gene and gene→chemical based on surface patterns."""
    for e in entities:
        s = normalize_span(e.get("text_span", ""))
        if e["label"] == "chemical" and GENE_LIKE.match(s):
            e["label"] = "gene"
        elif e["label"] == "gene" and CHEM_LIKE.search(s):
            e["label"] = "chemical"
    return entities


def postprocess_food_vs_supp(entities: list) -> list:
    """Remap food↔dietary supplement based on surface patterns."""
    for e in entities:
        s = normalize_span(e.get("text_span", ""))
        if any(w in s for w in FOOD_ANCHORS):
            if e["label"] in {"dietary supplement", "food"}: e["label"] = "food"
        elif any(w in s for w in SUPP_HARD):
            if e["label"] in {"dietary supplement", "food"}: e["label"] = "dietary supplement"
        elif e["label"] == "food" and any(w in s for w in SUPP_SOFT):
            e["label"] = "dietary supplement"
    return entities


def passes_threshold(e: dict, label_thresh: dict) -> bool:
    thr = label_thresh.get(e["label"], DEFAULT_THRESH)
    if e["label"] == "food" and "diet" in normalize_span(e.get("text_span", "")):
        thr = max(thr, 0.70)
    return e.get("score", 0.0) >= thr


def filter_by_threshold(entities: list, label_thresh: dict) -> list:
    return [e for e in entities if passes_threshold(e, label_thresh)]


def span_iou(a: dict, b: dict) -> float:
    inter = max(0, min(a["end_idx"], b["end_idx"]) - max(a["start_idx"], b["start_idx"]) + 1)
    if inter == 0: return 0.0
    la = a["end_idx"] - a["start_idx"] + 1
    lb = b["end_idx"] - b["start_idx"] + 1
    return inter / (la + lb - inter)


def any_overlap(ent: dict, kept: list, iou_thr: float = 0.5) -> bool:
    for k in kept:
        if k["location"] != ent["location"]: continue
        if ent["label"] == "food":
            if span_iou(ent, k) >= 0.85 and ent["label"] == k["label"]: return True
            continue
        if span_iou(ent, k) >= iou_thr: return True
    return False


def merge_two_pass(ents_high: list, ents_rec: list, recall_labels: set) -> list:
    kept = list(ents_high)
    for e in ents_rec:
        if e["label"] not in recall_labels: continue
        if not any_overlap(e, kept):        kept.append(e)
    return kept


def dedup_segment(entities: list) -> list:
    seen, out = set(), []
    for e in entities:
        key = (e["start_idx"], e["end_idx"], e["location"], e["label"])
        if key not in seen:
            seen.add(key)
            out.append(e)
    return out


def overlap_ratio(a: dict, b: dict) -> float:
    inter = max(0, min(a["end_idx"], b["end_idx"]) - max(a["start_idx"], b["start_idx"]) + 1)
    if inter == 0: return 0.0
    la = a["end_idx"] - a["start_idx"] + 1
    lb = b["end_idx"] - b["start_idx"] + 1
    return inter / min(la, lb)


def soft_overlap_prune(entities: list, same_label_only: bool = True,
                       ratio_thr: float = 0.85) -> list:
    """Keep the best span within near-duplicate clusters."""
    ents = sorted(entities, key=lambda x: (x["location"], x["start_idx"],
                                           -(x["end_idx"] - x["start_idx"])))
    kept = []
    for e in ents:
        drop = False
        for i, k in enumerate(kept):
            if k["location"] != e["location"]: continue
            if e["start_idx"] > k["end_idx"]:  continue  # no overlap possible
            if same_label_only and k["label"] != e["label"]: continue
            if overlap_ratio(k, e) >= ratio_thr:
                ks, es = k.get("score"), e.get("score")
                if ks is not None and es is not None:
                    kept[i] = e if es > ks else k
                else:
                    kept[i] = e if (e["end_idx"]-e["start_idx"]) > (k["end_idx"]-k["start_idx"]) else k
                drop = True; break
        if not drop:
            kept.append(e)
    return kept


print("Pipeline utility functions defined.")

Pipeline utility functions defined.


In [22]:
# ── Core decoder: BIO → entity spans with confidence scores ──────────────────

def predict_entities_with_scores(
    model, tokenizer, text: str,
    id2label: dict, label2id: dict,
    device, max_length: int = 512,
) -> list:
    """Run the model and return entities with median-token confidence scores."""
    if not text:
        return []

    enc = tokenizer(
        text, return_tensors="pt", truncation=True, padding=True,
        return_offsets_mapping=True, max_length=max_length,
    )
    offsets = enc.pop("offset_mapping")[0].cpu().numpy()
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits   = model(**enc).logits[0]
        probs    = torch.softmax(logits, dim=-1).cpu().numpy()
        pred_ids = np.argmax(probs, axis=-1)

    tags = [id2label[int(i)] for i in pred_ids]
    entities, current = [], None

    def _start(lbl, s, e_excl, ti):
        b_id  = label2id.get(f"B-{lbl}")
        tprob = float(probs[ti, b_id]) if b_id is not None else float(probs[ti].max())
        return {"start_idx": int(s), "end_idx": int(e_excl-1),
                "label": lbl, "text_span": text[s:e_excl], "_tp": [tprob]}

    def _extend(ent, e_excl, ti):
        i_id  = label2id.get(f"I-{ent['label']}")
        tprob = float(probs[ti, i_id]) if i_id is not None else float(probs[ti].max())
        ent["end_idx"]  = int(e_excl - 1)
        ent["text_span"] = text[ent["start_idx"]:e_excl]
        ent["_tp"].append(tprob)

    for ti, (tag, (s, e)) in enumerate(zip(tags, offsets)):
        s, e = int(s), int(e)
        if s == 0 and e == 0: continue
        if e <= s:             continue

        if tag.startswith("B-"):
            if current: entities.append(current)
            current = _start(tag[2:], s, e, ti)
        elif tag.startswith("I-"):
            lbl = tag[2:]
            if current is None:          current = _start(lbl, s, e, ti)  # BIO repair
            elif lbl != current["label"]: entities.append(current); current = _start(lbl, s, e, ti)
            else:                         _extend(current, e, ti)
        else:
            if current: entities.append(current); current = None

    if current: entities.append(current)

    for ent in entities:
        tp = ent.pop("_tp", [])
        ent["score"] = float(np.median(tp)) if tp else 0.0

    return entities


# ── Full two-pass predictor for one segment ───────────────────────────────────

def predict_segment_two_pass(
    model, tokenizer, text: str, location: str, device
) -> list:
    """Full inference pipeline for one title or abstract segment."""
    # Step 1 — raw BIO decode
    ents_raw = predict_entities_with_scores(
        model, tokenizer, text, id2label, label2id, device
    )

    # Step 2 — trim punctuation + assign location
    trimmed = []
    for e in ents_raw:
        e = trim_entity_span(e, text)
        if e and e.get("text_span"):
            e["location"] = location
            trimmed.append(e)

    # Step 3 — label remapping (before threshold)
    trimmed = postprocess_gene_vs_chemical(trimmed)
    trimmed = postprocess_food_vs_supp(trimmed)

    # Step 4 — Pass 1: high precision
    ents_high = filter_by_threshold(trimmed, LABEL_THRESH_HIGH)
    ents_high = apply_simple_filters(ents_high)

    # Step 5 — Pass 2: recall boost
    ents_rec = filter_by_threshold(trimmed, LABEL_THRESH_RECALL)
    ents_rec = apply_simple_filters(ents_rec)

    # Step 6 — Merge
    merged = merge_two_pass(ents_high, ents_rec, RECALL_LABELS)

    # Step 7 — Dedup + soft prune
    merged = dedup_segment(merged)
    merged = soft_overlap_prune(merged, same_label_only=True, ratio_thr=0.85)

    # Step 8 — Remove score field (not in submission format)
    for e in merged:
        e.pop("score", None)

    return merged


print("Inference functions defined.")

Inference functions defined.


## 5. Official evaluator — exact copy of `evaluate.py`

Two details that differ from naive implementations and **must** match exactly:
- `remove_duplicated_entities`: key is `(start_idx, end_idx, location)` — **no label**
- `remove_overlapping_entities`: overlap condition is `start_idx < current_end` — **strict** less-than

In [23]:
# ── Copied verbatim from evaluate.py ─────────────────────────────────────────

LEGAL_ENTITY_LABELS = [
    "anatomical location", "animal", "bacteria", "biomedical technique",
    "chemical", "DDF", "dietary supplement", "drug", "food", "gene",
    "human", "microbiome", "statistical technique"
]


def remove_duplicated_entities(predictions: dict) -> None:
    """Key = (start_idx, end_idx, location) — NO label field. In-place."""
    removed_count = 0
    for pmid in list(predictions.keys()):
        seen = set()
        deduped = []
        for ent in predictions[pmid]["entities"]:
            key = (ent["start_idx"], ent["end_idx"], ent["location"])
            if key not in seen:
                seen.add(key)
                deduped.append(ent)
            else:
                removed_count += 1
        predictions[pmid]["entities"] = deduped
    if removed_count > 0:
        print(f"=== Removed {removed_count} duplicated entities from predictions ===")


def remove_overlapping_entities(predictions: dict) -> None:
    """Overlap condition: start_idx < current_end  (strict). In-place."""
    removed_count = 0
    for pmid in list(predictions.keys()):
        original_len = len(predictions[pmid]['entities'])
        groups = {'title': [], 'abstract': []}
        for ent in predictions[pmid]['entities']:
            groups[ent["location"]].append(ent)
        keepers = set()
        for loc in groups:
            group = sorted(groups[loc], key=lambda e: e["start_idx"])
            clusters, cluster, current_end = [], [], None
            for ent in group:
                if not cluster:
                    cluster = [ent]
                    current_end = ent["end_idx"]
                else:
                    if ent["start_idx"] < current_end:   # ← strict, as in official
                        cluster.append(ent)
                        if ent["end_idx"] > current_end:
                            current_end = ent["end_idx"]
                    else:
                        clusters.append(cluster)
                        cluster = [ent]
                        current_end = ent["end_idx"]
            if cluster:
                clusters.append(cluster)
            for clust in clusters:
                longest = clust[0]
                max_len = longest["end_idx"] - longest["start_idx"]
                for ent in clust[1:]:
                    length = ent["end_idx"] - ent["start_idx"]
                    if length > max_len:
                        longest = ent
                        max_len = length
                keepers.add((longest["start_idx"], longest["end_idx"], longest["location"]))
        deduped = []
        for ent in predictions[pmid]['entities']:
            key = (ent["start_idx"], ent["end_idx"], ent["location"])
            if key in keepers:
                deduped.append(ent)
                keepers.remove(key)
        predictions[pmid]["entities"] = deduped
        removed_count += (original_len - len(deduped))
    if removed_count > 0:
        print(f"=== Removed {removed_count} overlapping entities ===")


def evaluate_official(predictions: dict, ground_truth: dict) -> dict:
    """
    Exact replica of eval_submission_NER() from evaluate.py.
    Works on an in-memory predictions dict (no file path needed).
    Returns a dict with macro/micro metrics + per-label breakdown.
    """
    # Work on a deep copy so the original predictions are not modified
    preds = copy.deepcopy(predictions)
    remove_duplicated_entities(preds)
    remove_overlapping_entities(preds)

    # Build gold lookup
    ground_truth_NER = {}
    count_annotated_entities_per_label = {}
    for pmid, article in ground_truth.items():
        if pmid not in ground_truth_NER:
            ground_truth_NER[pmid] = []
        for entity in article['entities']:
            start_idx = int(entity["start_idx"])
            end_idx   = int(entity["end_idx"])
            location  = str(entity["location"])
            text_span = str(entity["text_span"])
            label     = str(entity["label"])
            entry = (start_idx, end_idx, location, text_span, label)
            ground_truth_NER[pmid].append(entry)
            if label not in count_annotated_entities_per_label:
                count_annotated_entities_per_label[label] = 0
            count_annotated_entities_per_label[label] += 1

    count_predicted_entities_per_label = {label: 0 for label in count_annotated_entities_per_label}
    count_true_positives_per_label     = {label: 0 for label in count_annotated_entities_per_label}

    for pmid in preds.keys():
        entities = preds[pmid]['entities']
        for entity in entities:
            start_idx = int(entity["start_idx"])
            end_idx   = int(entity["end_idx"])
            location  = str(entity["location"])
            text_span = str(entity["text_span"])
            label     = str(entity["label"])
            if label not in LEGAL_ENTITY_LABELS:
                raise NameError(f'{pmid} - Illegal label {label} for entity: {entity}')
            if label in count_predicted_entities_per_label:
                count_predicted_entities_per_label[label] += 1
            entry = (start_idx, end_idx, location, text_span, label)
            if entry in ground_truth_NER[pmid]:
                count_true_positives_per_label[label] += 1

    count_annotated_entities = sum(count_annotated_entities_per_label.values())
    count_predicted_entities = sum(count_predicted_entities_per_label.values())
    count_true_positives     = sum(count_true_positives_per_label.values())

    micro_precision = count_true_positives / (count_predicted_entities + 1e-10)
    micro_recall    = count_true_positives / (count_annotated_entities  + 1e-10)
    micro_f1 = 2 * (micro_precision * micro_recall) / (micro_precision + micro_recall + 1e-10)

    precision = recall = f1 = 0
    n = 0
    per_label = {}
    for label in count_annotated_entities_per_label:
        n += 1
        cur_p  = count_true_positives_per_label[label] / (count_predicted_entities_per_label[label] + 1e-10)
        cur_r  = count_true_positives_per_label[label] / (count_annotated_entities_per_label[label]  + 1e-10)
        cur_f1 = 2 * (cur_p * cur_r) / (cur_p + cur_r + 1e-10)
        precision += cur_p
        recall    += cur_r
        f1        += cur_f1
        per_label[label] = {
            "P":    round(cur_p,  4),
            "R":    round(cur_r,  4),
            "F1":   round(cur_f1, 4),
            "TP":   count_true_positives_per_label[label],
            "pred": count_predicted_entities_per_label[label],
            "gold": count_annotated_entities_per_label[label],
        }
    precision /= n
    recall    /= n
    f1        /= n

    return {
        "macro_P":  round(precision,       4),
        "macro_R":  round(recall,           4),
        "macro_F1": round(f1,               4),
        "micro_P":  round(micro_precision,  4),
        "micro_R":  round(micro_recall,     4),
        "micro_F1": round(micro_f1,         4),
        "per_label": per_label,
    }


print("Official evaluator defined (evaluate.py — exact replica).")

Official evaluator defined (evaluate.py — exact replica).


## 6. Run inference on all models

Each model is loaded, run on the dev set, saved to disk, and evaluated.  
A model whose folder does not exist is **skipped** with a warning.

In [24]:
all_results = {}   # tag -> metrics dict

for tag, model_path in MODELS.items():
    print(f"\n{'='*60}")
    print(f"  Model: {tag}  —  {model_path.name}")
    print(f"{'='*60}")

    if not model_path.exists():
        print(f"  ✗ SKIPPED — folder not found: {model_path}")
        continue

    # ── Load model ────────────────────────────────────────────────────────────
    print("  Loading model…")
    tokenizer = AutoTokenizer.from_pretrained(str(model_path))
    model = AutoModelForTokenClassification.from_pretrained(
        str(model_path),
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id,
    ).to(DEVICE)
    model.eval()

    # ── Inference ─────────────────────────────────────────────────────────────
    predictions = {}
    for doc in tqdm(dev_documents, desc=f"  Inferring {tag}", leave=False):
        pmid     = doc["pmid"]
        location = doc["location"]
        text     = doc["text"]
        ents = predict_segment_two_pass(model, tokenizer, text, location, DEVICE)
        predictions.setdefault(pmid, {"entities": []})
        predictions[pmid]["entities"].extend(ents)

    total_pred = sum(len(v["entities"]) for v in predictions.values())
    print(f"  ✓ Predicted entities: {total_pred}")

    # ── Save prediction file ──────────────────────────────────────────────────
    out_path = PRED_FILES[tag]
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)
    print(f"  ✓ Saved: {out_path}")

    # ── Evaluate ──────────────────────────────────────────────────────────────
    metrics = evaluate_official(predictions, dev_data)
    all_results[tag] = metrics

    print(f"  Macro  P={metrics['macro_P']:.4f}  "
          f"R={metrics['macro_R']:.4f}  F1={metrics['macro_F1']:.4f}")
    print(f"  Micro  P={metrics['micro_P']:.4f}  "
          f"R={metrics['micro_R']:.4f}  F1={metrics['micro_F1']:.4f}")

    # ── Free GPU memory ───────────────────────────────────────────────────────
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n✓ All models processed.")


  Model: NB1b  —  bert_biomedbert_ner_twopass_gold_silver_bronze
  Loading model…


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9474.40it/s]


KeyboardInterrupt: 

## 7. Summary comparison table

In [ ]:
rows = []
for tag, m in all_results.items():
    rows.append({
        "Model": tag,
        "Macro-P":  m["macro_P"],
        "Macro-R":  m["macro_R"],
        "Macro-F1": m["macro_F1"],
        "Micro-P":  m["micro_P"],
        "Micro-R":  m["micro_R"],
        "Micro-F1": m["micro_F1"],
    })

df_summary = pd.DataFrame(rows).set_index("Model")

# Highlight best per column
def highlight_best(s):
    is_best = s == s.max()
    return ["font-weight: bold; background-color: #d4edda" if v else "" for v in is_best]

display(df_summary.style
        .apply(highlight_best)
        .format("{:.4f}")
        .set_caption("NER results — same inference pipeline on all models"))

print("\nBest Macro-F1:",
      df_summary["Macro-F1"].idxmax(),
      f"({df_summary['Macro-F1'].max():.4f})")
print("Best Micro-F1:",
      df_summary["Micro-F1"].idxmax(),
      f"({df_summary['Micro-F1'].max():.4f})")

## 8. Per-label F1 breakdown for all models

In [ ]:
label_rows = []
for tag, m in all_results.items():
    for lab, vals in m["per_label"].items():
        label_rows.append({"Model": tag, "Label": lab,
                           "P": vals["P"], "R": vals["R"], "F1": vals["F1"],
                           "TP": vals["TP"], "pred": vals["pred"], "gold": vals["gold"]})

df_labels = pd.DataFrame(label_rows)

# Pivot: rows=label, cols=model F1
df_pivot = df_labels.pivot(index="Label", columns="Model", values="F1").round(4)
# Add best column
df_pivot["Best"] = df_pivot.idxmax(axis=1)

print("\nPer-label F1 — all models:")
display(df_pivot.style
        .highlight_max(subset=[c for c in df_pivot.columns if c != "Best"],
                       color="#d4edda", axis=1)
        .format("{:.4f}", subset=[c for c in df_pivot.columns if c != "Best"]))

## 9. Save full results to JSON

In [ ]:
results_path = PRED_DIR / "all_models_results.json"
with results_path.open("w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
print(f"Full results saved to: {results_path}")